# Spaceship Titanic — упрощённая версия (только scikit-learn)
Запускайте ячейки сверху вниз. Нужны `train.csv`, `test.csv`, `sample_submission.csv` рядом с ноутбуком.

## 1. Данные

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score

train_raw = pd.read_csv("train.csv")
test_raw  = pd.read_csv("test.csv")
sub       = pd.read_csv("sample_submission.csv")

SPEND = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

## 2. Признаки
Train и test обрабатываются вместе (целевая переменная не используется), чтобы счётчики семей и групп были сопоставимы.

In [2]:
def prepare(df):
    df = df.copy()

    # группа из PassengerId
    df["Group"] = df["PassengerId"].str.split("_").str[0].astype(int)
    df["GroupSize"] = df.groupby("Group")["Group"].transform("count")

    # каюта
    df[["Deck", "Cabin_num", "Side"]] = df["Cabin"].str.split("/", expand=True)
    df["Cabin_num"] = pd.to_numeric(df["Cabin_num"])

    # пропуски берём у других членов группы (обычно одна планета и одна каюта)
    for col in ["HomePlanet", "Deck", "Side", "Cabin_num"]:
        df[col] = df[col].fillna(df.groupby("Group")[col].transform("first"))

    # размер семьи по фамилии
    df["Surname"] = df["Name"].str.split().str[-1]
    df["FamilySize"] = df.groupby("Surname")["Surname"].transform("count")

    # CryoSleep и траты
    cryo = df["CryoSleep"].astype(float)
    asleep = cryo == 1
    df.loc[asleep, SPEND] = df.loc[asleep, SPEND].fillna(0)
    spend_sum = df[SPEND].sum(axis=1)
    all_known = df[SPEND].notna().all(axis=1)
    miss = cryo.isna()
    cryo[miss & (spend_sum > 0)] = 0.0
    cryo[miss & all_known & (spend_sum == 0)] = 1.0
    df["CryoSleep"] = cryo

    df[SPEND] = df[SPEND].fillna(0)
    df["TotalSpend"] = df[SPEND].sum(axis=1)
    df["NoSpend"] = (df["TotalSpend"] == 0).astype(int)

    return df.drop(columns=["PassengerId", "Name", "Cabin", "Surname", "VIP"])

full = prepare(pd.concat([train_raw.drop(columns=["Transported"]), test_raw], ignore_index=True))
X      = full.iloc[:len(train_raw)].reset_index(drop=True)
X_test = full.iloc[len(train_raw):].reset_index(drop=True)
y      = train_raw["Transported"].astype(int)
groups = X["Group"]      # нужен только для честной кросс-валидации
X.head()

,HomePlanet,CryoSleep,Destination,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Group,GroupSize,Deck,Cabin_num,Side,FamilySize,TotalSpend,NoSpend
0,Europa,0.0,TRAPPIST-1e,39.0,0.0,0.0,0.0,0.0,0.0,1,1,B,0.0,P,3.0,0.0,1
1,Earth,0.0,TRAPPIST-1e,24.0,109.0,9.0,25.0,549.0,44.0,2,1,F,0.0,S,4.0,736.0,0
2,Europa,0.0,TRAPPIST-1e,58.0,43.0,3576.0,0.0,6715.0,49.0,3,2,A,0.0,S,7.0,10383.0,0
3,Europa,0.0,TRAPPIST-1e,33.0,0.0,1283.0,371.0,3329.0,193.0,3,2,A,0.0,S,7.0,5176.0,0
4,Earth,0.0,TRAPPIST-1e,16.0,303.0,70.0,151.0,565.0,2.0,4,1,F,1.0,S,9.0,1091.0,0


## 3. Модель и честная кросс-валидация
`StratifiedGroupKFold` держит группу целиком в одной части, поэтому оценка близка к тому, что будет на `test.csv`. Среднее по 3 запускам, чтобы меньше зависеть от случайности.

In [3]:
BASE_NUM = ["Age", "Cabin_num", "CryoSleep"] + SPEND
BASE_CAT = ["HomePlanet", "Side"]

def build(num_cols, cat_cols=BASE_CAT, clf=None):
    prep = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
        ("deck", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                           ("ord", OrdinalEncoder(handle_unknown="use_encoded_value",
                                                  unknown_value=-1))]), ["Deck"]),
    ])
    if clf is None:
        clf = HistGradientBoostingClassifier(
            max_iter=500, learning_rate=0.05, max_depth=6,
            l2_regularization=0.1, random_state=42)
    return Pipeline([("prep", prep), ("clf", clf)])

def cv_score(model, seeds=(0, 1, 2)):
    res = []
    for s in seeds:
        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=s)
        res.append(cross_val_score(model, X, y, cv=cv, groups=groups,
                                   scoring="accuracy", n_jobs=-1).mean())
    return np.mean(res), np.std(res)

## 4. Какие признаки помогают
Смотрите на число. Разница меньше 0.003–0.005 — это шум, берите более простой вариант.

In [4]:
variants = {
    "A база":                BASE_NUM,
    "B +GroupSize":          BASE_NUM + ["GroupSize"],
    "C +FamilySize":         BASE_NUM + ["FamilySize"],
    "D +TotalSpend,NoSpend": BASE_NUM + ["TotalSpend", "NoSpend"],
    "E все новые":           BASE_NUM + ["GroupSize", "FamilySize", "TotalSpend", "NoSpend"],
}
for name, cols in variants.items():
    m, s = cv_score(build(cols))
    print(f"{name:<24} {m:.4f} ± {s:.4f}")

m, s = cv_score(build(BASE_NUM, cat_cols=BASE_CAT + ["Destination"]))
print(f"{'A + Destination':<24} {m:.4f} ± {s:.4f}")

A база                   0.8033 ± 0.0018
B +GroupSize             0.8036 ± 0.0022
C +FamilySize            0.8016 ± 0.0012
D +TotalSpend,NoSpend    0.8073 ± 0.0015
E все новые              0.8036 ± 0.0018
A + Destination          0.8056 ± 0.0006


## 5. Модели и ансамбль
В `NUM` поставьте набор признаков, который выиграл в ячейке 4.

In [9]:
NUM = BASE_NUM + ["TotalSpend", "NoSpend"]

hgb_soft = HistGradientBoostingClassifier(
    max_iter=300, learning_rate=0.03, max_leaf_nodes=15,
    min_samples_leaf=30, l2_regularization=1.0, random_state=42)

rf = RandomForestClassifier(
    n_estimators=500, min_samples_leaf=3, max_features="sqrt",
    n_jobs=-1, random_state=42)

models = {
    "HGB (как раньше)": build(NUM),
    "HGB (мягче)":      build(NUM, clf=hgb_soft),
    "RandomForest":     build(NUM, clf=rf),
    "Ансамбль HGB+RF":  VotingClassifier(
        [("hgb", build(NUM, clf=hgb_soft)), ("rf", build(NUM, clf=rf))],
        voting="soft"),
}
for name, mdl in models.items():
    m, s = cv_score(mdl, seeds=(0,))
    print(f"{name:<20} {m:.4f}")

HGB (как раньше)     0.8052
HGB (мягче)          0.8095
RandomForest         0.8004
Ансамбль HGB+RF      0.8057


## 6. Итоговое обучение и submission
Выберите лучшую модель из ячейки 5 (имя в `models[...]`). Здесь она обучается на всём train, затем предсказывает для test.

In [ ]:
final_model = models["Ансамбль HGB+RF"]     # поменяйте на лучшую
final_model.fit(X, y)

sub["Transported"] = final_model.predict(X_test).astype(bool)
sub["PassengerId"] = test_raw["PassengerId"]
assert len(sub) == len(test_raw)
sub.to_csv("submission.csv", index=False)
print(sub["Transported"].value_counts())
sub.head()